In [ ]:
import os
import pandas as pd

file_path = "../data/multi_horizon_positive_temporal.csv"

if os.path.exists(file_path):
    saved_data = pd.read_csv(file_path)

    print("Saved extraction found!")
    print("Shape:", saved_data.shape)
    print("\nMissing values:")
    print(saved_data.isnull().sum())
else:
    print("Extraction file not found yet.")
    print("We can resume from the saved sample definitions.")

Saved extraction found!
Shape: (1125, 11)

Missing values:
latitude            0
longitude           0
reference_date      0
event_date          0
positive_horizon    0
RAIN_1D             0
RAIN_3D             0
RAIN_7D             0
RAIN_15D            0
RAIN_30D            0
LANDSLIDE           0
dtype: int64


In [ ]:
import pandas as pd

# Load the completed rainfall extraction
df = pd.read_csv("../data/multi_horizon_positive_temporal.csv")

df["reference_date"] = pd.to_datetime(df["reference_date"]).dt.normalize()
df["event_date"] = pd.to_datetime(df["event_date"]).dt.normalize()

# Build the unique event history from the extracted records
events = df[["latitude", "longitude", "event_date"]].drop_duplicates()

location_events = (
    events.groupby(["latitude", "longitude"])["event_date"]
    .apply(list)
    .to_dict()
)

# Calculate cumulative targets from each reference date
for horizon in [1, 2, 3]:
    labels = []

    for _, row in df.iterrows():
        key = (row["latitude"], row["longitude"])
        reference_date = row["reference_date"]

        future_events = [
            event_date
            for event_date in location_events[key]
            if reference_date < event_date
            <= reference_date + pd.Timedelta(days=horizon)
        ]

        labels.append(int(len(future_events) > 0))

    df[f"NEXT_{horizon}D"] = labels

print("Dataset shape:", df.shape)
print("\nTarget counts:")
print(df[["NEXT_1D", "NEXT_2D", "NEXT_3D"]].sum())

print("\nTarget combinations:")
print(df[["NEXT_1D", "NEXT_2D", "NEXT_3D"]].value_counts())

# Save this as a new file; preserve the original extraction
df.to_csv("../data/multi_horizon_positive_labeled.csv", index=False)
print("\nSaved: multi_horizon_positive_labeled.csv")

Dataset shape: (1125, 14)

Target counts:
NEXT_1D     375
NEXT_2D     750
NEXT_3D    1125
dtype: int64

Target combinations:
NEXT_1D  NEXT_2D  NEXT_3D
0        0        1          375
         1        1          375
1        1        1          375
Name: count, dtype: int64

Saved: multi_horizon_positive_labeled.csv


In [ ]:
import os

print("CSV files in data folder:")
print(sorted(f for f in os.listdir("../data") if f.endswith(".csv")))

if "background_locations" in globals():
    print("\nbackground_locations is still in memory:", background_locations.shape)
    background_locations.to_csv(
        "../data/background_locations.csv", index=False
    )
    print("Saved background_locations.csv")
elif "safe_negative_df" in globals():
    print("\nsafe_negative_df is still in memory:", safe_negative_df.shape)
    print("This may help recover the earlier background coordinates.")
else:
    print("\nBackground-location DataFrame not found in memory.")

CSV files in data folder:
['himalayan_event_rainfall.csv', 'himalayan_landslide_v1_dataset.csv', 'himalayan_temporal_dataset.csv', 'himalayan_temporal_dataset_v2.csv', 'matched_negative_temporal.csv', 'multi_horizon_positive_definitions.csv', 'multi_horizon_positive_labeled.csv', 'multi_horizon_positive_temporal.csv', 'negative_temporal_samples.csv', 'positive_temporal_samples.csv']

Background-location DataFrame not found in memory.


In [ ]:
import pandas as pd

files = [
    "../data/matched_negative_temporal.csv",
    "../data/negative_temporal_samples.csv",
]

for path in files:
    df = pd.read_csv(path)
    print(f"\n{path}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Unique coordinates:",
          df[["latitude", "longitude"]].drop_duplicates().shape[0])
    print("States column present:", "state" in df.columns)


../data/matched_negative_temporal.csv
Shape: (377, 12)
Columns: ['latitude', 'longitude', 'reference_date', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE_NEXT_3D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE']
Unique coordinates: 229
States column present: False

../data/negative_temporal_samples.csv
Shape: (374, 9)
Columns: ['latitude', 'longitude', 'reference_date', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE_NEXT_3D']
Unique coordinates: 374
States column present: False


In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil
import hashlib
import json
import zipfile

data_dir = Path("../data")
notebook_dir = Path("../notebooks")
backup_root = Path("../checkpoints")

# 1. Recover the missing background coordinate file
negative_path = data_dir / "negative_temporal_samples.csv"
if not negative_path.exists():
    raise FileNotFoundError(f"Required recovery source missing: {negative_path}")

negative = pd.read_csv(negative_path)
background_locations = (
    negative[["latitude", "longitude"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

if len(background_locations) != 374:
    raise ValueError(
        f"Expected 374 unique coordinates; found {len(background_locations)}. "
        "Stopped without saving a potentially incomplete recovery."
    )

background_path = data_dir / "background_locations.csv"
background_locations.to_csv(background_path, index=False)
print(f"Recovered background locations: {len(background_locations)}")

# 2. Create a dated checkpoint
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint = backup_root / f"checkpoint_{stamp}"
checkpoint.mkdir(parents=True, exist_ok=True)

# Copy all CSV files from data
copied_files = []
for source in sorted(data_dir.glob("*.csv")):
    destination = checkpoint / "data" / source.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    copied_files.append(destination)

# Copy saved notebooks, if present
if notebook_dir.exists():
    for source in notebook_dir.glob("*.ipynb"):
        destination = checkpoint / "notebooks" / source.name
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        copied_files.append(destination)

# 3. Write a manifest with file sizes, hashes, and CSV shapes
manifest = []
for path in copied_files:
    entry = {
        "file": str(path.relative_to(checkpoint)),
        "bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest()
    }

    if path.suffix.lower() == ".csv":
        frame = pd.read_csv(path)
        entry["rows"] = len(frame)
        entry["columns"] = frame.columns.tolist()

    manifest.append(entry)

manifest_path = checkpoint / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# 4. Make a portable ZIP backup
zip_path = backup_root / f"checkpoint_{stamp}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in checkpoint.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(checkpoint))

print("\nRECOVERY AND BACKUP COMPLETE")
print("Background file:", background_path.resolve())
print("Checkpoint folder:", checkpoint.resolve())
print("ZIP backup:", zip_path.resolve())
print("Files backed up:", len(copied_files))
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))

Recovered background locations: 374

RECOVERY AND BACKUP COMPLETE
Background file: C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\data\background_locations.csv
Checkpoint folder: C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\checkpoints\checkpoint_20261004_170111
ZIP backup: C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\checkpoints\checkpoint_20261004_170111.zip
Files backed up: 14
ZIP size (MB): 1.29


In [ ]:
from pathlib import Path
import pandas as pd
import json
import hashlib

data_dir = Path("../data")
backup_root = Path("../checkpoints")

# 1. Verify recovered background coordinates
background = pd.read_csv(data_dir / "background_locations.csv")
negative = pd.read_csv(data_dir / "negative_temporal_samples.csv")

print("BACKGROUND COORDINATE AUDIT")
print("Rows:", len(background))
print("Unique coordinates:",
      background[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Missing values:", background.isnull().sum().sum())
print("Matches earlier negative coordinates:",
      set(map(tuple, background[["latitude", "longitude"]].to_numpy()))
      == set(map(tuple, negative[["latitude", "longitude"]].drop_duplicates().to_numpy())))

# 2. Verify the latest checkpoint manifest and file hashes
checkpoints = sorted(
    p for p in backup_root.glob("checkpoint_*")
    if p.is_dir()
)
if not checkpoints:
    raise FileNotFoundError("No checkpoint folder found.")

checkpoint = checkpoints[-1]
manifest_path = checkpoint / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

errors = []
for item in manifest:
    file_path = checkpoint / item["file"]
    if not file_path.exists():
        errors.append(f"Missing: {item['file']}")
        continue

    digest = hashlib.sha256(file_path.read_bytes()).hexdigest()
    if digest != item["sha256"]:
        errors.append(f"Hash mismatch: {item['file']}")

print("\nCHECKPOINT AUDIT")
print("Checkpoint:", checkpoint.name)
print("Manifest entries:", len(manifest))
print("Verification errors:", errors if errors else "None")
print("ZIP backups:", len(list(backup_root.glob("checkpoint_*.zip"))))

BACKGROUND COORDINATE AUDIT
Rows: 374
Unique coordinates: 374
Missing values: 0
Matches earlier negative coordinates: True

CHECKPOINT AUDIT
Checkpoint: checkpoint_20261004_170111
Manifest entries: 14
Verification errors: None
ZIP backups: 1


In [ ]:
import pandas as pd

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
negative = pd.read_csv("../data/matched_negative_temporal.csv")

positive["reference_date"] = pd.to_datetime(
    positive["reference_date"]
).dt.normalize()

negative["reference_date"] = pd.to_datetime(
    negative["reference_date"]
).dt.normalize()

positive_dates = set(positive["reference_date"])
negative_dates = set(negative["reference_date"])

print("TEMPORAL DATA COVERAGE")
print("Positive rows:", len(positive))
print("Positive unique dates:", len(positive_dates))
print("Existing negative rows:", len(negative))
print("Existing negative unique dates:", len(negative_dates))
print("Dates already covered by negatives:",
      len(positive_dates & negative_dates))
print("Positive dates missing from negative data:",
      len(positive_dates - negative_dates))

print("\nPositive reference-date range:",
      positive["reference_date"].min().date(), "to",
      positive["reference_date"].max().date())

print("Negative reference-date range:",
      negative["reference_date"].min().date(), "to",
      negative["reference_date"].max().date())

TEMPORAL DATA COVERAGE
Positive rows: 1125
Positive unique dates: 477
Existing negative rows: 377
Existing negative unique dates: 212
Dates already covered by negatives: 212
Positive dates missing from negative data: 265

Positive reference-date range: 2007-07-27 to 2018-10-08
Negative reference-date range: 2007-07-27 to 2018-10-06


In [ ]:
import pandas as pd
import numpy as np

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
existing = pd.read_csv("../data/matched_negative_temporal.csv")
background = pd.read_csv("../data/background_locations.csv")

for df in [positive, existing]:
    df["reference_date"] = pd.to_datetime(
        df["reference_date"]
    ).dt.normalize()

rng = np.random.default_rng(42)
rows = []
reused_dates = 0
new_dates = 0

# One control for each unique positive reference date
for ref_date in sorted(positive["reference_date"].unique()):

    available = existing[
        existing["reference_date"] == ref_date
    ]

    if not available.empty:
        # Reuse a saved control for this date
        chosen = available.iloc[0]
        lat, lon = chosen["latitude"], chosen["longitude"]
        reused_dates += 1
    else:
        # Select a previously screened background location
        chosen_idx = rng.integers(len(background))
        chosen = background.iloc[chosen_idx]
        lat, lon = chosen["latitude"], chosen["longitude"]
        new_dates += 1

    rows.append({
        "latitude": lat,
        "longitude": lon,
        "reference_date": ref_date,
        "LANDSLIDE": 0
    })

controls = pd.DataFrame(rows)
controls.to_csv(
    "../data/multi_horizon_negative_definitions.csv",
    index=False
)

print("Control definitions saved.")
print("Rows:", len(controls))
print("Unique dates:", controls["reference_date"].nunique())
print("Existing dates reused:", reused_dates)
print("New dates needing feature extraction:", new_dates)
print("Missing values:", controls.isnull().sum().sum())
print("File: multi_horizon_negative_definitions.csv")

Control definitions saved.
Rows: 477
Unique dates: 477
Existing dates reused: 212
New dates needing feature extraction: 265
Missing values: 0
File: multi_horizon_negative_definitions.csv


In [ ]:
import pandas as pd
import numpy as np

controls = pd.read_csv("../data/multi_horizon_negative_definitions.csv")
events = pd.read_csv("../data/himalayan_event_rainfall.csv")

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()
events["event_date"] = pd.to_datetime(
    events["event_date"]
).dt.normalize()

def haversine_km(lat, lon, lats, lons):
    lat1 = np.radians(lat)
    lon1 = np.radians(lon)
    lat2 = np.radians(np.asarray(lats, dtype=float))
    lon2 = np.radians(np.asarray(lons, dtype=float))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

event_lats = events["latitude"].to_numpy(dtype=float)
event_lons = events["longitude"].to_numpy(dtype=float)

nearest_distances = []
future_event_flags = []

for _, control in controls.iterrows():
    distances = haversine_km(
        float(control["latitude"]),
        float(control["longitude"]),
        event_lats,
        event_lons
    )

    future_mask = (
        (events["event_date"] > control["reference_date"]) &
        (events["event_date"] <=
         control["reference_date"] + pd.Timedelta(days=3))
    ).to_numpy()

    nearest_distances.append(float(np.min(distances)))

    # Is any event within 1 km during the next 3 days?
    future_event_flags.append(
        bool(np.any(distances[future_mask] <= 1.0))
    )

controls["nearest_event_km"] = nearest_distances
controls["event_within_1km_next_3d"] = future_event_flags

print("Controls checked:", len(controls))
print("Within 1 km of a future recorded event:",
      int(controls["event_within_1km_next_3d"].sum()))
print("Median nearest-event distance (km):",
      round(controls["nearest_event_km"].median(), 3))
print("Minimum nearest-event distance (km):",
      round(controls["nearest_event_km"].min(), 3))

controls.to_csv(
    "../data/multi_horizon_negative_definitions_audited.csv",
    index=False
)
print("Saved audited control definitions.")

Controls checked: 477
Within 1 km of a future recorded event: 4
Median nearest-event distance (km): 0.0
Minimum nearest-event distance (km): 0.0
Saved audited control definitions.


In [ ]:
controls = pd.read_csv(
    "../data/multi_horizon_negative_definitions_audited.csv"
)

problem_controls = controls[
    controls["event_within_1km_next_3d"] == True
]

print(problem_controls[
    ["latitude", "longitude", "reference_date",
     "nearest_event_km"]
].to_string(index=False))

 latitude  longitude reference_date  nearest_event_km
30.719800  77.169500     2010-09-18               0.0
30.430500  78.080300     2014-07-18               0.0
31.103127  77.203555     2017-09-01               0.0
30.722031  78.445173     2018-05-31               0.0


In [ ]:
import pandas as pd
import numpy as np

background = pd.read_csv("../data/background_locations.csv")
events = pd.read_csv("../data/himalayan_event_rainfall.csv")

def haversine_km(lat, lon, lats, lons):
    lat1, lon1 = np.radians([lat, lon])
    lat2 = np.radians(np.asarray(lats, dtype=float))
    lon2 = np.radians(np.asarray(lons, dtype=float))
    a = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    )
    return 6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

event_lats = events["latitude"].to_numpy()
event_lons = events["longitude"].to_numpy()

nearest = []
for _, loc in background.iterrows():
    d = haversine_km(
        loc["latitude"], loc["longitude"], event_lats, event_lons
    )
    nearest.append(float(d.min()))

nearest = np.array(nearest)

print("Background locations:", len(background))
print("Nearest-event distance, minimum (km):", round(nearest.min(), 3))
print("Locations farther than 1 km:", int((nearest > 1).sum()))
print("Locations farther than 2 km:", int((nearest > 2).sum()))
print("Locations farther than 5 km:", int((nearest > 5).sum()))

Background locations: 374
Nearest-event distance, minimum (km): 0.0
Locations farther than 1 km: 0
Locations farther than 2 km: 0
Locations farther than 5 km: 0


In [ ]:
import os
import pandas as pd
import numpy as np

files = [
    "../data/background_locations_v2.csv",
    "../data/multi_horizon_negative_definitions_v2.csv"
]

for file in files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        print(f"\n{os.path.basename(file)}")
        print("Rows:", len(df))
        print("Missing values:", int(df.isna().sum().sum()))
        print("Unique coordinates:", len(
            df[["latitude", "longitude"]].drop_duplicates()
        ))
    else:
        print(f"\nNOT FOUND: {file}")


NOT FOUND: ../data/background_locations_v2.csv

NOT FOUND: ../data/multi_horizon_negative_definitions_v2.csv


In [ ]:
import ee
import pandas as pd
import numpy as np

ee.Initialize(project="integral-server-484809-p2")

# Load the study region
states = ee.FeatureCollection("FAO/GAUL/2015/level1")
region = states.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "India"),
        ee.Filter.inList(
            "ADM1_NAME",
            ["Himachal Pradesh", "Uttarakhand"]
        )
    )
)

# Load the historical event locations
events = pd.read_csv("../data/himalayan_event_rainfall.csv")
positive = pd.read_csv(
    "../data/multi_horizon_positive_labeled.csv"
)

event_coords = events[
    ["latitude", "longitude"]
].drop_duplicates().to_numpy(dtype=float)

dates = sorted(
    pd.to_datetime(positive["reference_date"])
    .dt.normalize()
    .unique()
)
required = len(dates)

def haversine_km(lat, lon, coords):
    lat1, lon1 = np.radians([lat, lon])
    lat2 = np.radians(coords[:, 0])
    lon2 = np.radians(coords[:, 1])
    a = (
        np.sin((lat2-lat1)/2)**2
        + np.cos(lat1)*np.cos(lat2)
        * np.sin((lon2-lon1)/2)**2
    )
    return 6371 * 2 * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )

safe = []
seen = set()

# Generate batches until enough safe points are found
for batch in range(20):
    points = ee.FeatureCollection.randomPoints(
        region=region.geometry(),
        points=2500,
        seed=420 + batch,
        maxError=100
    ).getInfo()

    for feature in points["features"]:
        lon, lat = feature["geometry"]["coordinates"]
        key = (round(lat, 6), round(lon, 6))

        if key in seen:
            continue
        seen.add(key)

        if np.all(haversine_km(lat, lon, event_coords) > 1.0):
            safe.append({
                "latitude": lat,
                "longitude": lon
            })

    print(f"Batch {batch + 1}: {len(safe)} eligible locations")

    if len(safe) >= required:
        break

if len(safe) < required:
    raise RuntimeError(
        f"Found {len(safe)} locations; need {required}. "
        "No output files were saved."
    )

background_v2 = pd.DataFrame(safe[:required])
background_v2.to_csv(
    "../data/background_locations_v2.csv",
    index=False
)

controls_v2 = background_v2.copy()
controls_v2["reference_date"] = dates
controls_v2["LANDSLIDE"] = 0
controls_v2.to_csv(
    "../data/multi_horizon_negative_definitions_v2.csv",
    index=False
)

print("\nSaved background locations:", len(background_v2))
print("Saved control definitions:", len(controls_v2))
print(
    "Minimum event distance (km):",
    round(min(
        haversine_km(r.latitude, r.longitude, event_coords).min()
        for r in background_v2.itertuples()
    ), 3)
)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python
C:\Users\imman\AppData\Roaming\Python\Python311\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


Batch 1: 2483 eligible locations

Saved background locations: 477
Saved control definitions: 477
Minimum event distance (km): 1.39


In [ ]:
import pandas as pd
import numpy as np

controls = pd.read_csv(
    "../data/multi_horizon_negative_definitions_v2.csv"
)
events = pd.read_csv(
    "../data/himalayan_event_rainfall.csv"
)

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()
events["event_date"] = pd.to_datetime(
    events["event_date"]
).dt.normalize()

def haversine_km(lat, lon, lats, lons):
    lat1, lon1 = np.radians([lat, lon])
    lat2 = np.radians(np.asarray(lats, dtype=float))
    lon2 = np.radians(np.asarray(lons, dtype=float))
    a = (
        np.sin((lat2-lat1)/2)**2
        + np.cos(lat1)*np.cos(lat2)
        * np.sin((lon2-lon1)/2)**2
    )
    return 6371 * 2 * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )

event_lats = events["latitude"].to_numpy()
event_lons = events["longitude"].to_numpy()

flagged = []

for i, row in controls.iterrows():
    future = events[
        (events["event_date"] > row["reference_date"]) &
        (events["event_date"] <=
         row["reference_date"] + pd.Timedelta(days=3))
    ]

    if future.empty:
        continue

    distances = haversine_km(
        row["latitude"], row["longitude"],
        future["latitude"].to_numpy(),
        future["longitude"].to_numpy()
    )

    if np.any(distances <= 1.0):
        flagged.append(i)

print("Controls checked:", len(controls))
print("Controls near a future recorded event:", len(flagged))

if flagged:
    print(controls.loc[
        flagged,
        ["latitude", "longitude", "reference_date"]
    ].to_string(index=False))
else:
    print("PASS: No control is within 1 km of a recorded event in its next 3 days.")

Controls checked: 477
Controls near a future recorded event: 0
PASS: No control is within 1 km of a recorded event in its next 3 days.


In [ ]:
import os
import ee
import pandas as pd

ee.Initialize(project="integral-server-484809-p2")

controls = pd.read_csv(
    "../data/multi_horizon_negative_definitions_v2.csv"
)

print("Control rows:", len(controls))
print("Missing values:", int(controls.isna().sum().sum()))

print("\nRequired source files:")
for filename in [
    "himalayan_landslide_v1_dataset.csv",
    "multi_horizon_positive_labeled.csv",
    "himalayan_event_rainfall.csv"
]:
    path = "../data/" + filename
    print(filename, "FOUND" if os.path.exists(path) else "MISSING")

print("\nControl date range:")
print(controls["reference_date"].min(), "to",
      controls["reference_date"].max())

Control rows: 477
Missing values: 0

Required source files:
himalayan_landslide_v1_dataset.csv FOUND
multi_horizon_positive_labeled.csv FOUND
himalayan_event_rainfall.csv FOUND

Control date range:
2007-07-27 to 2018-10-08


In [ ]:

import ee
import pandas as pd
import numpy as np
import os
import time

ee.Initialize(project="integral-server-484809-p2")

controls = pd.read_csv(
    "../data/multi_horizon_negative_definitions_v2.csv"
)
controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()

output_path = "../data/multi_horizon_negative_features_v2.csv"
progress_path = "../data/multi_horizon_negative_features_progress.csv"

# Static environmental layers
dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem)
worldcover = ee.ImageCollection(
    "ESA/WorldCover/v200"
).first().select("Map")

# Daily rainfall collection
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

def extract_one(lat, lon, ref_date):
    point = ee.Geometry.Point([lon, lat])
    end = ee.Date(ref_date.strftime("%Y-%m-%d"))

    rain_bands = []
    for days in [1, 3, 7, 15, 30]:
        rainfall = (
            chirps
            .filterDate(end.advance(-days, "day"), end)
            .sum()
            .rename(f"RAIN_{days}D")
        )
        rain_bands.append(rainfall)

    rain_image = ee.Image.cat(rain_bands)

    rain_values = rain_image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=5000,
        maxPixels=1_000_000
    ).getInfo()

    static_image = ee.Image.cat([
        dem.rename("ELEVATION"),
        slope.rename("SLOPE"),
        worldcover.rename("LANDCOVER_CODE")
    ])

    static_values = static_image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30,
        maxPixels=1_000_000
    ).getInfo()

    result = {
        "latitude": lat,
        "longitude": lon,
        "reference_date": ref_date.strftime("%Y-%m-%d"),
        **rain_values,
        **static_values
    }

    return result

# Resume from completed rows, if available
if os.path.exists(progress_path):
    completed = pd.read_csv(progress_path)
    completed_keys = set(
        zip(
            completed["latitude"].round(6),
            completed["longitude"].round(6),
            completed["reference_date"]
        )
    )
else:
    completed = pd.DataFrame()
    completed_keys = set()

pending = []
for _, row in controls.iterrows():
    key = (
        round(row["latitude"], 6),
        round(row["longitude"], 6),
        row["reference_date"].strftime("%Y-%m-%d")
    )
    if key not in completed_keys:
        pending.append(row)

print("Total controls:", len(controls))
print("Already extracted:", len(completed))
print("Remaining:", len(pending))

for start in range(0, len(pending), 25):
    batch = pending[start:start + 25]
    results = []

    for row in batch:
        results.append(
            extract_one(
                float(row["latitude"]),
                float(row["longitude"]),
                row["reference_date"]
            )
        )

    batch_df = pd.DataFrame(results)
    completed = pd.concat(
        [completed, batch_df], ignore_index=True
    )
    completed = completed.drop_duplicates(
        subset=["latitude", "longitude", "reference_date"],
        keep="last"
    )
    completed.to_csv(progress_path, index=False)

    print(
        f"Saved {len(completed)} / {len(controls)} controls"
    )

# Final feature file
completed.to_csv(output_path, index=False)

print("\nExtraction finished.")
print("Rows:", len(completed))
print("Missing values:", int(completed.isna().sum().sum()))
print("Saved:", output_path)


Total controls: 477
Already extracted: 0
Remaining: 477
Saved 25 / 477 controls
Saved 50 / 477 controls
Saved 75 / 477 controls
Saved 100 / 477 controls
Saved 125 / 477 controls
Saved 150 / 477 controls
Saved 175 / 477 controls
Saved 200 / 477 controls
Saved 225 / 477 controls
Saved 250 / 477 controls
Saved 275 / 477 controls
Saved 300 / 477 controls
Saved 325 / 477 controls
Saved 350 / 477 controls
Saved 375 / 477 controls
Saved 400 / 477 controls
Saved 425 / 477 controls
Saved 450 / 477 controls
Saved 475 / 477 controls
Saved 477 / 477 controls

Extraction finished.
Rows: 477
Missing values: 0
Saved: ../data/multi_horizon_negative_features_v2.csv


In [ ]:
import pandas as pd
import numpy as np

negative = pd.read_csv(
    "../data/multi_horizon_negative_features_v2.csv"
)
positive = pd.read_csv(
    "../data/multi_horizon_positive_labeled.csv"
)

print("NEGATIVE FEATURES")
print("Shape:", negative.shape)
print("Missing values:", negative.isna().sum().sum())
print("\nFeature ranges:")
print(negative[
    ["ELEVATION", "SLOPE", "RAIN_1D", "RAIN_3D",
     "RAIN_7D", "RAIN_15D", "RAIN_30D"]
].describe().round(2))

print("\nPOSITIVE FEATURES")
print("Shape:", positive.shape)
print("Missing values:", positive.isna().sum().sum())

required = [
    "ELEVATION", "SLOPE", "LANDCOVER_CODE",
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

print("\nRequired features present in negative data:")
print(set(required).issubset(negative.columns))

print("\nRequired features present in positive data:")
print(set(required).issubset(positive.columns))

NEGATIVE FEATURES
Shape: (477, 11)
Missing values: 0

Feature ranges:
       ELEVATION   SLOPE  RAIN_1D  RAIN_3D  RAIN_7D  RAIN_15D  RAIN_30D
count     477.00  477.00   477.00   477.00   477.00    477.00    477.00
mean     2444.04   24.52     9.79    28.69    64.37    136.74    254.80
std      1704.70   13.98    21.85    38.62    59.81    102.59    174.01
min       202.00    0.00     0.00     0.00     0.00      0.00      0.00
25%      1079.00   14.86     0.00     0.00    11.89     48.08    105.03
50%      1979.00   25.55     0.00    12.01    51.74    122.57    232.23
75%      3895.00   34.60    11.97    45.62    99.59    206.65    387.75
max      6098.00   67.40   259.49   259.49   320.97    560.98    864.07

POSITIVE FEATURES
Shape: (1125, 14)
Missing values: 0

Required features present in negative data:
True

Required features present in positive data:
False


In [ ]:
import pandas as pd

negative = pd.read_csv(
    "../data/multi_horizon_negative_features_v2.csv"
)
positive = pd.read_csv(
    "../data/multi_horizon_positive_labeled.csv"
)

print("NEGATIVE columns:")
print(negative.columns.tolist())

print("\nPOSITIVE columns:")
print(positive.columns.tolist())

print("\nColumns missing from positive:")
print(sorted(set(negative.columns) - set(positive.columns)))

print("\nColumns missing from negative:")
print(sorted(set(positive.columns) - set(negative.columns)))

NEGATIVE columns:
['latitude', 'longitude', 'reference_date', 'RAIN_15D', 'RAIN_1D', 'RAIN_30D', 'RAIN_3D', 'RAIN_7D', 'ELEVATION', 'LANDCOVER_CODE', 'SLOPE']

POSITIVE columns:
['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D']

Columns missing from positive:
['ELEVATION', 'LANDCOVER_CODE', 'SLOPE']

Columns missing from negative:
['LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D', 'event_date', 'positive_horizon']


In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

positive = pd.read_csv(
    "../data/multi_horizon_positive_labeled.csv"
)

static = pd.read_csv(
    "../data/himalayan_landslide_v1_dataset.csv"
)

# Keep only the environmental features we need
static_features = static[
    [
        "LATITUDE", "LONGITUDE",
        "ELEVATION", "SLOPE", "LANDCOVER_CODE"
    ]
].drop_duplicates(subset=["LATITUDE", "LONGITUDE"])

# Find the nearest static-data coordinate for each positive event
coords_static = np.radians(
    static_features[["LATITUDE", "LONGITUDE"]].to_numpy()
)
coords_positive = np.radians(
    positive[["latitude", "longitude"]].to_numpy()
)

nn = NearestNeighbors(
    n_neighbors=1,
    metric="haversine",
    algorithm="ball_tree"
)
nn.fit(coords_static)

distances, indices = nn.kneighbors(coords_positive)

matched = static_features.iloc[indices[:, 0]][
    ["ELEVATION", "SLOPE", "LANDCOVER_CODE"]
].reset_index(drop=True)

positive_fixed = pd.concat(
    [positive.reset_index(drop=True), matched],
    axis=1
)

output_path = "../data/multi_horizon_positive_features_v2.csv"
positive_fixed.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", positive_fixed.shape)
print("Missing values:", int(positive_fixed.isna().sum().sum()))
print(
    "Nearest-point distance, median (km):",
    round(float(np.median(distances[:, 0]) * 6371), 3)
)
print(
    "Nearest-point distance, maximum (km):",
    round(float(np.max(distances[:, 0]) * 6371), 3)
)

Saved: ../data/multi_horizon_positive_features_v2.csv
Shape: (1125, 17)
Missing values: 0
Nearest-point distance, median (km): 0.705
Nearest-point distance, maximum (km): 4.238


In [ ]:
import pandas as pd
import numpy as np

positive = pd.read_csv(
    "../data/multi_horizon_positive_features_v2.csv"
)
negative = pd.read_csv(
    "../data/multi_horizon_negative_features_v2.csv"
)

features = [
    "ELEVATION", "SLOPE", "LANDCOVER_CODE",
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

print("Positive shape:", positive.shape)
print("Negative shape:", negative.shape)

print("\nMissing feature columns:")
print("Positive:", [c for c in features if c not in positive.columns])
print("Negative:", [c for c in features if c not in negative.columns])

print("\nMissing feature values:")
print("Positive:", int(positive[features].isna().sum().sum()))
print("Negative:", int(negative[features].isna().sum().sum()))

print("\nInvalid negative rainfall ordering (should be zero):")
rain = negative[["RAIN_1D", "RAIN_3D", "RAIN_7D",
                 "RAIN_15D", "RAIN_30D"]]
print("Rows:", int((
    (rain["RAIN_1D"] > rain["RAIN_3D"]) |
    (rain["RAIN_3D"] > rain["RAIN_7D"]) |
    (rain["RAIN_7D"] > rain["RAIN_15D"]) |
    (rain["RAIN_15D"] > rain["RAIN_30D"])
).sum()))

Positive shape: (1125, 17)
Negative shape: (477, 11)

Missing feature columns:
Positive: []
Negative: []

Missing feature values:
Positive: 0
Negative: 0

Invalid negative rainfall ordering (should be zero):
Rows: 0


In [1]:
import pandas as pd

positive = pd.read_csv(
    "../data/multi_horizon_positive_features_v2.csv"
)

positive["reference_date"] = pd.to_datetime(
    positive["reference_date"]
).dt.normalize()

print("Positive rows:", len(positive))
print("Unique reference dates:", positive["reference_date"].nunique())
print("\nRows per reference date:")
print(positive.groupby("reference_date").size().describe())

print("\nPositive target totals:")
print(positive[["NEXT_1D", "NEXT_2D", "NEXT_3D"]].sum())

print("\nPositive target patterns:")
print(
    positive.groupby(["NEXT_1D", "NEXT_2D", "NEXT_3D"])
    .size()
    .to_string()
)

Positive rows: 1125
Unique reference dates: 477

Rows per reference date:
count    477.000000
mean       2.358491
std        2.740797
min        1.000000
25%        1.000000
50%        1.000000
75%        3.000000
max       28.000000
dtype: float64

Positive target totals:
NEXT_1D     375
NEXT_2D     750
NEXT_3D    1125
dtype: int64

Positive target patterns:
NEXT_1D  NEXT_2D  NEXT_3D
0        0        1          375
         1        1          375
1        1        1          375


In [4]:
import pandas as pd
import numpy as np

positive = pd.read_csv(
    "../data/multi_horizon_positive_features_v2.csv"
)

controls = pd.read_csv(
    "../data/multi_horizon_negative_features_v2.csv"
)

positive["reference_date"] = pd.to_datetime(
    positive["reference_date"]
).dt.normalize()

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()

features = [
    "ELEVATION",
    "SLOPE",
    "LANDCOVER_CODE",
    "RAIN_1D",
    "RAIN_3D",
    "RAIN_7D",
    "RAIN_15D",
    "RAIN_30D"
]

# --------------------------------------------------
# 1. Create one negative control for every
#    positive reference-date/horizon row
# --------------------------------------------------

negative = positive[
    ["reference_date"]
].copy()

# Match each date to its safe control location/features
negative = negative.merge(
    controls[
        ["reference_date", "latitude", "longitude"] + features
    ],
    on="reference_date",
    how="left",
    validate="many_to_one"
)

# Negative targets
negative["LANDSLIDE"] = 0
negative["NEXT_1D"] = 0
negative["NEXT_2D"] = 0
negative["NEXT_3D"] = 0

# --------------------------------------------------
# 2. Select the common modeling columns
# --------------------------------------------------

positive_model = positive[
    ["latitude", "longitude", "reference_date"] +
    features +
    ["NEXT_1D", "NEXT_2D", "NEXT_3D"]
].copy()

negative_model = negative[
    ["latitude", "longitude", "reference_date"] +
    features +
    ["NEXT_1D", "NEXT_2D", "NEXT_3D"]
].copy()

# Add class indicator
positive_model["LANDSLIDE"] = 1
negative_model["LANDSLIDE"] = 0

# --------------------------------------------------
# 3. Combine
# --------------------------------------------------

model_data = pd.concat(
    [positive_model, negative_model],
    ignore_index=True
)

model_data = model_data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# --------------------------------------------------
# 4. Save
# --------------------------------------------------

output_path = "../data/himalayan_temporal_model_dataset_v1.csv"

model_data.to_csv(
    output_path,
    index=False
)

# --------------------------------------------------
# 5. Basic validation
# --------------------------------------------------

print("FINAL DATASET")
print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))

print("\nClass distribution:")
print(model_data["LANDSLIDE"].value_counts())

print("\nTarget totals:")
print(
    model_data[
        ["NEXT_1D", "NEXT_2D", "NEXT_3D"]
    ].sum()
)

print("\nMissing values:")
print(int(model_data.isna().sum().sum()))

print("\nDuplicate rows:")
print(int(model_data.duplicated().sum()))

print("\nSaved:")
print(output_path)

FINAL DATASET
Rows: 2250
Columns: 15

Class distribution:
LANDSLIDE
1    1125
0    1125
Name: count, dtype: int64

Target totals:
NEXT_1D     375
NEXT_2D     750
NEXT_3D    1125
dtype: int64

Missing values:
0

Duplicate rows:
648

Saved:
../data/himalayan_temporal_model_dataset_v1.csv


In [5]:
import pandas as pd

df = pd.read_csv(
    "../data/himalayan_temporal_model_dataset_v1.csv"
)

print("Total rows:", len(df))
print("Exact duplicate rows:", df.duplicated().sum())

# Count duplicate groups
duplicates = (
    df[df.duplicated(keep=False)]
    .groupby(list(df.columns))
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("\nDuplicate groups:", len(duplicates))
print("\nTop duplicate groups:")
print(duplicates.head(10).to_string(index=False))

# Check whether duplicates are across different classes
key = [
    "latitude",
    "longitude",
    "reference_date",
    "ELEVATION",
    "SLOPE",
    "LANDCOVER_CODE",
    "RAIN_1D",
    "RAIN_3D",
    "RAIN_7D",
    "RAIN_15D",
    "RAIN_30D"
]

class_counts = (
    df.groupby(key)["LANDSLIDE"]
    .nunique()
)

print(
    "\nFeature-identical groups containing both classes:",
    int((class_counts > 1).sum())
)

Total rows: 2250
Exact duplicate rows: 648

Duplicate groups: 223

Top duplicate groups:
 latitude  longitude reference_date  ELEVATION     SLOPE  LANDCOVER_CODE   RAIN_1D    RAIN_3D    RAIN_7D   RAIN_15D   RAIN_30D  NEXT_1D  NEXT_2D  NEXT_3D  LANDSLIDE  count
30.823393  79.513579     2010-09-17       4848 26.171894              60  0.000000   0.000000   0.000000  72.701385 119.594364        0        0        0          0     28
32.517091  78.086755     2010-09-18       5157 30.406609              60  0.000000   0.000000   0.000000   6.681337   9.482821        0        0        0          0     25
29.987828  80.477496     2010-09-16       4003 24.670900             100  0.000000   0.000000  49.271332 260.856840 465.764368        0        0        0          0     23
30.261967  77.986462     2013-06-14        644  5.432300              10 60.635403 161.694401 188.805277 207.291909 223.170286        0        0        0          0     15
30.567187  78.997283     2013-06-15       2160 25.4

In [6]:
import pandas as pd

df = pd.read_csv(
    "../data/himalayan_temporal_model_dataset_v1.csv"
)

negative = df[df["LANDSLIDE"] == 0]

location_date_counts = (
    negative
    .groupby(["latitude", "longitude", "reference_date"])
    .size()
    .sort_values(ascending=False)
)

print("Negative rows:", len(negative))
print("Unique negative location-date pairs:",
      len(location_date_counts))

print("\nRepeated negative location-date pairs:",
      int((location_date_counts > 1).sum()))

print("\nMaximum reuse of one location-date:",
      int(location_date_counts.max()))

print("\nReuse distribution:")
print(location_date_counts.describe())

Negative rows: 1125
Unique negative location-date pairs: 477

Repeated negative location-date pairs: 223

Maximum reuse of one location-date: 28

Reuse distribution:
count    477.000000
mean       2.358491
std        2.740797
min        1.000000
25%        1.000000
50%        1.000000
75%        3.000000
max       28.000000
dtype: float64


In [8]:
import ee
import pandas as pd

ee.Initialize(project="integral-server-484809-p2")

states = ee.FeatureCollection(
    "FAO/GAUL/2015/level1"
)

study_region = states.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "India"),
        ee.Filter.inList(
            "ADM1_NAME",
            ["Himachal Pradesh", "Uttarakhand"]
        )
    )
)

positive = pd.read_csv(
    "../data/multi_horizon_positive_features_v2.csv"
)

required = len(positive)

print("Negative locations required:", required)

all_candidates = []

# 3 batches × 4000 = 12,000 candidates
for batch in range(3):

    points = ee.FeatureCollection.randomPoints(
        region=study_region.geometry(),
        points=4000,
        seed=20261005 + batch,
        maxError=100
    )

    data = points.getInfo()

    for feature in data["features"]:
        lon, lat = feature["geometry"]["coordinates"]

        all_candidates.append({
            "latitude": lat,
            "longitude": lon
        })

    print(
        f"Batch {batch + 1}/3 complete — "
        f"{len(all_candidates)} candidates"
    )

candidates_df = pd.DataFrame(all_candidates)

candidates_df = candidates_df.drop_duplicates(
    subset=["latitude", "longitude"]
).reset_index(drop=True)

print("\nTotal unique candidates:", len(candidates_df))

candidates_df.to_csv(
    "../data/negative_candidate_pool_v2.csv",
    index=False
)

print("Saved:")
print("../data/negative_candidate_pool_v2.csv")

Negative locations required: 1125
Batch 1/3 complete — 4000 candidates
Batch 2/3 complete — 8000 candidates
Batch 3/3 complete — 12000 candidates

Total unique candidates: 12000
Saved:
../data/negative_candidate_pool_v2.csv


In [9]:
import pandas as pd
import numpy as np

candidates = pd.read_csv(
    "../data/negative_candidate_pool_v2.csv"
)

events = pd.read_csv(
    "../data/himalayan_event_rainfall.csv"
)

def haversine_km(lat, lon, lats, lons):
    lat1 = np.radians(lat)
    lon1 = np.radians(lon)

    lat2 = np.radians(np.asarray(lats, dtype=float))
    lon2 = np.radians(np.asarray(lons, dtype=float))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 6371.0 * 2 * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )

event_coords = events[
    ["latitude", "longitude"]
].drop_duplicates().to_numpy(dtype=float)

event_lats = event_coords[:, 0]
event_lons = event_coords[:, 1]

nearest_distances = []

for _, row in candidates.iterrows():

    distances = haversine_km(
        row["latitude"],
        row["longitude"],
        event_lats,
        event_lons
    )

    nearest_distances.append(distances.min())

candidates["nearest_event_km"] = nearest_distances

safe_candidates = candidates[
    candidates["nearest_event_km"] > 1.0
].copy()

safe_candidates = safe_candidates.sort_values(
    "nearest_event_km",
    ascending=False
).reset_index(drop=True)

print("Total candidates:", len(candidates))
print("Safe candidates (>1 km):", len(safe_candidates))
print(
    "Minimum safe distance:",
    round(safe_candidates["nearest_event_km"].min(), 3),
    "km"
)

safe_candidates.to_csv(
    "../data/negative_safe_candidate_pool_v2.csv",
    index=False
)

print("Saved:")
print("../data/negative_safe_candidate_pool_v2.csv")

Total candidates: 12000
Safe candidates (>1 km): 11901
Minimum safe distance: 1.014 km
Saved:
../data/negative_safe_candidate_pool_v2.csv


In [10]:
import pandas as pd
import numpy as np

safe = pd.read_csv(
    "../data/negative_safe_candidate_pool_v2.csv"
)

positive = pd.read_csv(
    "../data/multi_horizon_positive_features_v2.csv"
)

positive["reference_date"] = pd.to_datetime(
    positive["reference_date"]
).dt.normalize()

# We need exactly one unique negative location
# for every positive forecasting row.
required = len(positive)

# Randomize candidate order
safe = safe.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Select first 1125 unique locations
negative_locations = safe.iloc[:required].copy()

# Assign the positive reference dates
negative_locations["reference_date"] = (
    positive["reference_date"].to_numpy()
)

negative_locations["LANDSLIDE"] = 0

negative_locations.to_csv(
    "../data/negative_location_definitions_v3.csv",
    index=False
)

print("Negative locations:", len(negative_locations))
print(
    "Unique coordinates:",
    len(
        negative_locations[
            ["latitude", "longitude"]
        ].drop_duplicates()
    )
)
print(
    "Unique location-date pairs:",
    len(
        negative_locations[
            ["latitude", "longitude", "reference_date"]
        ].drop_duplicates()
    )
)
print(
    "Reference dates:",
    negative_locations["reference_date"].nunique()
)

print("\nSaved:")
print("../data/negative_location_definitions_v3.csv")

Negative locations: 1125
Unique coordinates: 1125
Unique location-date pairs: 1125
Reference dates: 477

Saved:
../data/negative_location_definitions_v3.csv


In [11]:
import pandas as pd
import numpy as np

controls = pd.read_csv(
    "../data/negative_location_definitions_v3.csv"
)

events = pd.read_csv(
    "../data/himalayan_event_rainfall.csv"
)

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()

events["event_date"] = pd.to_datetime(
    events["event_date"]
).dt.normalize()


def haversine_km(lat, lon, lats, lons):
    lat1 = np.radians(lat)
    lon1 = np.radians(lon)

    lat2 = np.radians(np.asarray(lats, dtype=float))
    lon2 = np.radians(np.asarray(lons, dtype=float))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 6371.0 * 2 * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )


event_lats = events["latitude"].to_numpy(dtype=float)
event_lons = events["longitude"].to_numpy(dtype=float)

flagged = []

for i, row in controls.iterrows():

    # Events occurring during the following 3 days
    future = events[
        (events["event_date"] > row["reference_date"]) &
        (
            events["event_date"]
            <= row["reference_date"] + pd.Timedelta(days=3)
        )
    ]

    if future.empty:
        continue

    distances = haversine_km(
        row["latitude"],
        row["longitude"],
        future["latitude"].to_numpy(dtype=float),
        future["longitude"].to_numpy(dtype=float)
    )

    if np.any(distances <= 1.0):
        flagged.append(i)


print("Controls checked:", len(controls))
print("Unique coordinates:", len(
    controls[["latitude", "longitude"]].drop_duplicates()
))
print("Controls near a future recorded event:", len(flagged))

if flagged:
    print("\nPROBLEMATIC CONTROLS:")
    print(
        controls.loc[
            flagged,
            ["latitude", "longitude", "reference_date"]
        ].to_string(index=False)
    )
else:
    print(
        "\nPASS: No negative control is within "
        "1 km of a recorded event during its next 3 days."
    )

# Save the audit result
controls["future_event_within_1km"] = False
controls.loc[flagged, "future_event_within_1km"] = True

controls.to_csv(
    "../data/negative_location_definitions_v3_audited.csv",
    index=False
)

print("\nSaved:")
print("../data/negative_location_definitions_v3_audited.csv")

Controls checked: 1125
Unique coordinates: 1125
Controls near a future recorded event: 0

PASS: No negative control is within 1 km of a recorded event during its next 3 days.

Saved:
../data/negative_location_definitions_v3_audited.csv


In [2]:
import ee
import pandas as pd
import os

ee.Authenticate()
ee.Initialize(project="integral-server-484809-p2")



controls = pd.read_csv(
    "../data/negative_location_definitions_v4.csv"
)

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()

output_path = "../data/negative_features_v4.csv"
progress_path = "../data/negative_features_v4_progress.csv"

# --------------------------------------------------
# Earth Engine datasets
# --------------------------------------------------

dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem)

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)

chirps = ee.ImageCollection(
    "UCSB-CHG/CHIRPS/DAILY"
)

# --------------------------------------------------
# Extract features for one control
# --------------------------------------------------

def extract_features(lat, lon, ref_date):

    point = ee.Geometry.Point([lon, lat])

    end = ee.Date(
        ref_date.strftime("%Y-%m-%d")
    )

    rainfall_images = []

    for days in [1, 3, 7, 15, 30]:

        rainfall = (
            chirps
            .filterDate(
                end.advance(-days, "day"),
                end
            )
            .sum()
            .rename(f"RAIN_{days}D")
        )

        rainfall_images.append(rainfall)

    rainfall_image = ee.Image.cat(
        rainfall_images
    )

    rainfall_values = rainfall_image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=5000,
        maxPixels=1_000_000
    ).getInfo()

    static_image = ee.Image.cat([
        dem.rename("ELEVATION"),
        slope.rename("SLOPE"),
        worldcover.rename("LANDCOVER_CODE")
    ])

    static_values = static_image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30,
        maxPixels=1_000_000
    ).getInfo()

    return {
        "latitude": lat,
        "longitude": lon,
        "reference_date": ref_date.strftime("%Y-%m-%d"),
        **rainfall_values,
        **static_values
    }


# --------------------------------------------------
# Resume if extraction was already started
# --------------------------------------------------

if os.path.exists(progress_path):

    completed = pd.read_csv(
        progress_path
    )

    completed_keys = set(
        zip(
            completed["latitude"].round(6),
            completed["longitude"].round(6),
            completed["reference_date"]
        )
    )

else:

    completed = pd.DataFrame()
    completed_keys = set()


pending = []

for _, row in controls.iterrows():

    key = (
        round(row["latitude"], 6),
        round(row["longitude"], 6),
        row["reference_date"].strftime("%Y-%m-%d")
    )

    if key not in completed_keys:
        pending.append(row)


print("Total controls:", len(controls))
print("Already extracted:", len(completed))
print("Remaining:", len(pending))


# --------------------------------------------------
# Extract in batches
# --------------------------------------------------

for start in range(0, len(pending), 25):

    batch = pending[start:start + 25]

    results = []

    for row in batch:

        result = extract_features(
            float(row["latitude"]),
            float(row["longitude"]),
            row["reference_date"]
        )

        results.append(result)

    batch_df = pd.DataFrame(results)

    completed = pd.concat(
        [completed, batch_df],
        ignore_index=True
    )

    completed = completed.drop_duplicates(
        subset=[
            "latitude",
            "longitude",
            "reference_date"
        ],
        keep="last"
    )

    completed.to_csv(
        progress_path,
        index=False
    )

    print(
        f"Saved {len(completed)} / {len(controls)}"
    )


# --------------------------------------------------
# Final file
# --------------------------------------------------

completed.to_csv(
    output_path,
    index=False
)

print("\nEXTRACTION COMPLETE")
print("Rows:", len(completed))
print(
    "Missing values:",
    int(completed.isna().sum().sum())
)
print("Saved:", output_path)


Successfully saved authorization token.


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Total controls: 1125
Already extracted: 0
Remaining: 1125
Saved 25 / 1125
Saved 50 / 1125
Saved 75 / 1125
Saved 100 / 1125
Saved 125 / 1125
Saved 150 / 1125
Saved 175 / 1125
Saved 200 / 1125
Saved 225 / 1125
Saved 250 / 1125
Saved 275 / 1125
Saved 300 / 1125
Saved 325 / 1125
Saved 350 / 1125
Saved 375 / 1125
Saved 400 / 1125
Saved 425 / 1125
Saved 450 / 1125
Saved 475 / 1125
Saved 500 / 1125
Saved 525 / 1125
Saved 550 / 1125
Saved 575 / 1125
Saved 600 / 1125
Saved 625 / 1125
Saved 650 / 1125
Saved 675 / 1125
Saved 700 / 1125
Saved 725 / 1125
Saved 750 / 1125
Saved 775 / 1125
Saved 800 / 1125
Saved 825 / 1125
Saved 850 / 1125
Saved 875 / 1125
Saved 900 / 1125
Saved 925 / 1125
Saved 950 / 1125
Saved 975 / 1125
Saved 1000 / 1125
Saved 1025 / 1125
Saved 1050 / 1125
Saved 1075 / 1125
Saved 1100 / 1125
Saved 1125 / 1125

EXTRACTION COMPLETE
Rows: 1125
Missing values: 0
Saved: ../data/negative_features_v4.csv


In [3]:
import pandas as pd

controls = pd.read_csv("../data/negative_location_definitions_v4.csv")
features = pd.read_csv("../data/negative_features_v4.csv")

print("Expected control samples:", len(controls))
print("Extracted feature rows:", len(features))
print("Missing values:", features.isnull().sum().sum())

print("Unique extracted location-date pairs:",
      features[["latitude", "longitude", "reference_date"]]
      .drop_duplicates().shape[0])

print("\nFeature columns:")
print(features.columns.tolist())

Expected control samples: 1125
Extracted feature rows: 1125
Missing values: 0
Unique extracted location-date pairs: 1125

Feature columns:
['latitude', 'longitude', 'reference_date', 'RAIN_15D', 'RAIN_1D', 'RAIN_30D', 'RAIN_3D', 'RAIN_7D', 'ELEVATION', 'LANDCOVER_CODE', 'SLOPE']


In [13]:
import pandas as pd

neg = pd.read_csv("../data/negative_features_v3.csv")

print("Shape:", neg.shape)
print("\nMissing values:")
print(neg.isna().sum())

print("\nDuplicate rows:", neg.duplicated().sum())

print("\nUnique coordinates:",
      neg[["latitude", "longitude"]].drop_duplicates().shape[0])

print("\nColumns:")
print(neg.columns.tolist())

Shape: (1125, 11)

Missing values:
latitude          0
longitude         0
reference_date    0
RAIN_15D          0
RAIN_1D           0
RAIN_30D          0
RAIN_3D           0
RAIN_7D           0
ELEVATION         0
LANDCOVER_CODE    0
SLOPE             0
dtype: int64

Duplicate rows: 0

Unique coordinates: 1125

Columns:
['latitude', 'longitude', 'reference_date', 'RAIN_15D', 'RAIN_1D', 'RAIN_30D', 'RAIN_3D', 'RAIN_7D', 'ELEVATION', 'LANDCOVER_CODE', 'SLOPE']


In [14]:
import pandas as pd

pos = pd.read_csv("../data/multi_horizon_positive_labeled.csv")

print("Shape:", pos.shape)
print("\nColumns:")
print(pos.columns.tolist())

print("\nMissing values:")
print(pos.isna().sum())

print("\nTarget counts:")
print(pos[["NEXT_1D", "NEXT_2D", "NEXT_3D"]].sum())

Shape: (1125, 14)

Columns:
['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D']

Missing values:
latitude            0
longitude           0
reference_date      0
event_date          0
positive_horizon    0
RAIN_1D             0
RAIN_3D             0
RAIN_7D             0
RAIN_15D            0
RAIN_30D            0
LANDSLIDE           0
NEXT_1D             0
NEXT_2D             0
NEXT_3D             0
dtype: int64

Target counts:
NEXT_1D     375
NEXT_2D     750
NEXT_3D    1125
dtype: int64


In [16]:
import pandas as pd

pos = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
pos_env = pd.read_csv("../data/multi_horizon_positive_features_v2.csv")

print("Positive temporal:", pos.shape)
print("Positive environmental:", pos_env.shape)

print("\nEnvironmental columns:")
print(pos_env.columns.tolist())

Positive temporal: (1125, 14)
Positive environmental: (1125, 17)

Environmental columns:
['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE']


In [17]:
import pandas as pd

# Load complete positive dataset
pos = pd.read_csv("../data/multi_horizon_positive_features_v2.csv")

# Load complete negative dataset
neg = pd.read_csv("../data/negative_features_v3.csv")

# Add targets to negative samples
neg["LANDSLIDE"] = 0
neg["NEXT_1D"] = 0
neg["NEXT_2D"] = 0
neg["NEXT_3D"] = 0

# Positive samples already have all targets
pos["LANDSLIDE"] = 1

# Columns needed for modeling
columns = [
    "latitude",
    "longitude",
    "reference_date",
    "RAIN_1D",
    "RAIN_3D",
    "RAIN_7D",
    "RAIN_15D",
    "RAIN_30D",
    "ELEVATION",
    "SLOPE",
    "LANDCOVER_CODE",
    "LANDSLIDE",
    "NEXT_1D",
    "NEXT_2D",
    "NEXT_3D"
]

pos_final = pos[columns].copy()
neg_final = neg[columns].copy()

# Combine
final_df = pd.concat(
    [pos_final, neg_final],
    ignore_index=True
)

print("Final shape:", final_df.shape)

print("\nClass counts:")
print(final_df["LANDSLIDE"].value_counts())

print("\nTarget counts:")
print(final_df[["NEXT_1D", "NEXT_2D", "NEXT_3D"]].sum())

print("\nMissing values:")
print(final_df.isna().sum())

Final shape: (2250, 15)

Class counts:
LANDSLIDE
1    1125
0    1125
Name: count, dtype: int64

Target counts:
NEXT_1D     375
NEXT_2D     750
NEXT_3D    1125
dtype: int64

Missing values:
latitude          0
longitude         0
reference_date    0
RAIN_1D           0
RAIN_3D           0
RAIN_7D           0
RAIN_15D          0
RAIN_30D          0
ELEVATION         0
SLOPE             0
LANDCOVER_CODE    0
LANDSLIDE         0
NEXT_1D           0
NEXT_2D           0
NEXT_3D           0
dtype: int64


In [18]:
final_df.to_csv(
    "../data/himalayan_temporal_final_v3.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", final_df.shape)

Saved successfully.
Shape: (2250, 15)


In [19]:
# Reload the saved final dataset
import pandas as pd

df = pd.read_csv("../data/himalayan_temporal_final_v3.csv")

print("Shape:", df.shape)

# 1. Exact duplicate rows
print("\nExact duplicate rows:", df.duplicated().sum())

# 2. Duplicate location + reference date
location_date_dupes = df.duplicated(
    subset=["latitude", "longitude", "reference_date"]
).sum()

print("Duplicate location-date rows:", location_date_dupes)

# 3. Check target consistency for identical location-date samples
target_cols = ["NEXT_1D", "NEXT_2D", "NEXT_3D"]

conflicting = (
    df.groupby(["latitude", "longitude", "reference_date"])[target_cols]
      .nunique()
)

conflicting_groups = (conflicting > 1).any(axis=1).sum()

print("Location-date groups with conflicting targets:",
      conflicting_groups)

# 4. Check whether positive and negative samples share locations
positive_locations = set(
    zip(
        df.loc[df["LANDSLIDE"] == 1, "latitude"],
        df.loc[df["LANDSLIDE"] == 1, "longitude"]
    )
)

negative_locations = set(
    zip(
        df.loc[df["LANDSLIDE"] == 0, "latitude"],
        df.loc[df["LANDSLIDE"] == 0, "longitude"]
    )
)

overlap = positive_locations.intersection(negative_locations)

print("Locations appearing in BOTH classes:", len(overlap))

Shape: (2250, 15)

Exact duplicate rows: 0
Duplicate location-date rows: 0
Location-date groups with conflicting targets: 0
Locations appearing in BOTH classes: 0


In [20]:
print("Reference date range:")
print(df["reference_date"].min(), "→", df["reference_date"].max())

print("\nUnique reference dates:")
print(df["reference_date"].nunique())

print("\nSamples by year:")
print(
    pd.to_datetime(df["reference_date"])
    .dt.year
    .value_counts()
    .sort_index()
)

print("\nLatitude range:")
print(df["latitude"].min(), "→", df["latitude"].max())

print("\nLongitude range:")
print(df["longitude"].min(), "→", df["longitude"].max())

Reference date range:
2007-07-27 → 2018-10-08

Unique reference dates:
477

Samples by year:
reference_date
2007     42
2008     54
2009     48
2010    528
2011    294
2012     90
2013    480
2014     42
2015    234
2016    144
2017    138
2018    156
Name: count, dtype: int64

Latitude range:
28.748794616239543 → 33.157201026802575

Longitude range:
75.82966094270475 → 80.8576


In [21]:
# Convert date
df["reference_date"] = pd.to_datetime(df["reference_date"])

print("Samples by year and class:\n")

year_class = pd.crosstab(
    df["reference_date"].dt.year,
    df["LANDSLIDE"]
)

print(year_class)

print("\nPositive samples by year:")
print(
    df[df["LANDSLIDE"] == 1]["reference_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

print("\nNegative samples by year:")
print(
    df[df["LANDSLIDE"] == 0]["reference_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Samples by year and class:

LANDSLIDE         0    1
reference_date          
2007             21   21
2008             27   27
2009             24   24
2010            264  264
2011            147  147
2012             45   45
2013            240  240
2014             21   21
2015            117  117
2016             72   72
2017             69   69
2018             78   78

Positive samples by year:
reference_date
2007     21
2008     27
2009     24
2010    264
2011    147
2012     45
2013    240
2014     21
2015    117
2016     72
2017     69
2018     78
Name: count, dtype: int64

Negative samples by year:
reference_date
2007     21
2008     27
2009     24
2010    264
2011    147
2012     45
2013    240
2014     21
2015    117
2016     72
2017     69
2018     78
Name: count, dtype: int64


In [22]:
# Final feature sanity check

feature_cols = [
    "RAIN_1D",
    "RAIN_3D",
    "RAIN_7D",
    "RAIN_15D",
    "RAIN_30D",
    "ELEVATION",
    "SLOPE",
    "LANDCOVER_CODE"
]

print("FEATURE RANGES\n")

for col in feature_cols:
    print(
        f"{col:18} "
        f"min={df[col].min():.2f}   "
        f"max={df[col].max():.2f}   "
        f"mean={df[col].mean():.2f}"
    )

print("\nRainfall ordering violations:")

violations = (
    (df["RAIN_1D"] > df["RAIN_3D"]) |
    (df["RAIN_3D"] > df["RAIN_7D"]) |
    (df["RAIN_7D"] > df["RAIN_15D"]) |
    (df["RAIN_15D"] > df["RAIN_30D"])
).sum()

print("Violations:", violations)

FEATURE RANGES

RAIN_1D            min=0.00   max=231.39   mean=11.04
RAIN_3D            min=0.00   max=396.04   mean=34.76
RAIN_7D            min=0.00   max=488.28   mean=79.83
RAIN_15D           min=0.00   max=816.05   mean=169.17
RAIN_30D           min=0.00   max=1249.35   mean=303.29
ELEVATION          min=192.00   max=6330.00   mean=2136.26
SLOPE              min=0.00   max=74.97   mean=25.51
LANDCOVER_CODE     min=10.00   max=100.00   mean=30.62

Rainfall ordering violations:
Violations: 0
